# 🎬 날씨별 영화관

**Open-Meteo + TMDB + Python + Gradio**로 만드는 영화 추천 프로젝트입니다.

사용자가 도시를 선택하면 현재 날씨를 조회하고, 날씨에 어울리는 장르의 영화를 TMDB에서 가져와 포스터 형태로 보여줍니다.

## 1. 패키지 설치

In [25]:
!pip install -U requests gradio python-dotenv

'pip'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


## 2. TMDB API 키 설정

프로젝트 폴더에 `.env` 파일을 만들고 다음 내용을 넣으세요.

```text
TMDB_API_KEY=여기에_TMDB_Read_Access_Token
```

In [26]:
import os
import requests
import gradio as gr
from dotenv import load_dotenv

load_dotenv()

TMDB_API_KEY = os.getenv("TMDB_API_KEY")

if not TMDB_API_KEY:
    raise ValueError("TMDB_API_KEY가 없습니다. .env 파일을 확인하세요.")

print("✅ TMDB API 키 로드 완료")

✅ TMDB API 키 로드 완료


## 3. 도시 데이터

Open-Meteo에 전달할 위도와 경도를 도시별로 정의합니다.

In [27]:
CITIES = {
    "서울": {"latitude": 37.5665, "longitude": 126.9780},
    "부산": {"latitude": 35.1796, "longitude": 129.0756},
    "대구": {"latitude": 35.8714, "longitude": 128.6014},
    "인천": {"latitude": 37.4563, "longitude": 126.7052},
    "광주": {"latitude": 35.1595, "longitude": 126.8526},
    "대전": {"latitude": 36.3504, "longitude": 127.3845},
    "제주": {"latitude": 33.4996, "longitude": 126.5312},
    "도쿄": {"latitude": 35.6762, "longitude": 139.6503},
    "뉴욕": {"latitude": 40.7128, "longitude": -74.0060},
    "런던": {"latitude": 51.5074, "longitude": -0.1278},
}

## 4. 날씨 → 영화 장르 매핑

WMO weather code를 프로젝트에서 사용하기 쉬운 5가지 상태로 단순화합니다.

In [28]:
WEATHER_INFO = {
    "clear": {
        "name": "맑음",
        "emoji": "☀️",
        "message": "햇살 좋은 날에는 신나는 영화가 잘 어울려요!",
        "genres": [12, 35, 28],
    },
    "cloudy": {
        "name": "흐림",
        "emoji": "☁️",
        "message": "조금 흐린 날에는 분위기 있는 영화가 어울려요.",
        "genres": [18, 53, 9648],
    },
    "rain": {
        "name": "비",
        "emoji": "🌧️",
        "message": "비 오는 날에는 집에서 영화 한 편 어떠세요?",
        "genres": [18, 10749, 9648],
    },
    "snow": {
        "name": "눈",
        "emoji": "❄️",
        "message": "눈 오는 날에는 따뜻한 판타지 영화가 좋아요!",
        "genres": [14, 16, 12],
    },
    "storm": {
        "name": "폭풍",
        "emoji": "⛈️",
        "message": "강한 날씨에는 긴장감 넘치는 영화가 어울려요!",
        "genres": [53, 27, 28],
    },
}

def get_weather_type(weather_code):
    if weather_code == 0:
        return "clear"
    if weather_code in [1, 2, 3]:
        return "cloudy"
    if weather_code in [51, 53, 55, 56, 57, 61, 63, 65, 66, 67, 80, 81, 82]:
        return "rain"
    if weather_code in [71, 73, 75, 77, 85, 86]:
        return "snow"
    if weather_code in [95, 96, 99]:
        return "storm"
    return "cloudy"

## 5. Open-Meteo API 함수

In [29]:
def get_weather(city):
    if city not in CITIES:
        raise ValueError(f"지원하지 않는 도시입니다: {city}")

    location = CITIES[city]

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "current": "temperature_2m,weather_code",
        "timezone": "auto",
    }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()

    return response.json()["current"]

# 테스트
print(get_weather("서울"))

{'time': '2026-09-03T17:15', 'interval': 900, 'temperature_2m': 30.0, 'weather_code': 0}


## 6. TMDB API 함수

날씨별 추천 장르를 OR 조건으로 검색합니다.

In [30]:
def get_movies(genre_ids, limit=5):
    url = "https://api.themoviedb.org/3/discover/movie"

    headers = {
        "Authorization": f"Bearer {TMDB_API_KEY}",
        "accept": "application/json",
    }

    params = {
        "with_genres": "|".join(map(str, genre_ids)),
        "sort_by": "popularity.desc",
        "vote_count.gte": 100,
        "language": "ko-KR",
        "page": 1,
    }

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=10,
    )
    response.raise_for_status()

    return response.json().get("results", [])[:limit]

## 7. 추천 함수

In [31]:
def recommend_movies(city):
    try:
        current = get_weather(city)

        temperature = current["temperature_2m"]
        weather_code = current["weather_code"]

        weather_type = get_weather_type(weather_code)
        info = WEATHER_INFO[weather_type]

        movies = get_movies(info["genres"], limit=5)

        weather_markdown = f'''
## {info["emoji"]} {city}의 현재 날씨: {info["name"]}

### 🌡️ {temperature:.1f}°C

> {info["message"]}

**추천 영화 장르:** {", ".join(map(str, info["genres"]))}
'''

        gallery = []

        for movie in movies:
            poster_path = movie.get("poster_path")
            if not poster_path:
                continue

            poster_url = f"https://image.tmdb.org/t/p/w500{poster_path}"
            title = (
                movie.get("title")
                or movie.get("original_title")
                or "제목 없음"
            )
            rating = movie.get("vote_average", 0)

            gallery.append(
                (poster_url, f"{title} ⭐ {rating:.1f}")
            )

        if not gallery:
            weather_markdown += "\n\n⚠️ 표시할 영화 포스터가 없습니다."

        return weather_markdown, gallery

    except requests.RequestException as e:
        return f"## ❌ API 요청 오류\n\n`{e}`", []

    except Exception as e:
        return f"## ❌ 오류\n\n`{e}`", []

## 8. Gradio UI

도시를 선택한 뒤 **영화 추천받기** 버튼을 누르면 날씨와 영화 포스터가 표시됩니다.

In [32]:
custom_css = """
.gradio-container {
    max-width: 1100px !important;
}

h1 {
    text-align: center;
}
"""

with gr.Blocks(
    title="날씨별 영화관",
    theme=gr.themes.Soft(),
    css=custom_css,
) as demo:

    gr.Markdown(
        """
        # 🎬 날씨별 영화관
        ### 오늘 날씨에 어울리는 영화를 추천해드립니다.

        **Open-Meteo 🌤️ + TMDB 🎥**
        """
    )

    with gr.Row():
        city_dropdown = gr.Dropdown(
            choices=list(CITIES.keys()),
            value="서울",
            label="📍 도시 선택",
            scale=2,
        )

        recommend_button = gr.Button(
            "🎬 영화 추천받기",
            variant="primary",
            scale=1,
        )

    weather_output = gr.Markdown()

    movie_gallery = gr.Gallery(
        label="🍿 오늘의 추천 영화",
        columns=5,
        rows=1,
        height="auto",
        object_fit="cover",
    )

    recommend_button.click(
        fn=recommend_movies,
        inputs=city_dropdown,
        outputs=[weather_output, movie_gallery],
    )

    gr.Markdown(
        """
        ---
        **Data:** Open-Meteo · TMDB
        """
    )

demo.launch()

C:\Users\soldesk\AppData\Local\Temp\ipykernel_14284\3467002441.py:11: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## 9. 다음 개선 아이디어

- 도시 직접 검색
- 오늘/내일/주말 예보 기반 추천
- 영화 줄거리와 개봉일 표시
- 장르 이름을 한글로 표시
- 평점순 / 인기순 선택
- 추천 영화 개수 선택
- 영화 상세 정보 패널